# SQL Agent Workflow Demo

参考 `sql-agent.ipynb`，把「自然语言 → SQL → 自然语言」拆成 7 个可控步骤：

**数据库解析 → table 选择 → SQL 生成 → SQL 校验 → SQL 改写 → SQL 执行 → 结果汇总**

本 notebook 只做演示，核心逻辑在 `sql_agent/` 包里。运行前先执行 `pip install openai`，并把 API Key 写入 `.env`。


In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from sql_agent import DBParser, Settings, SQLAgentWorkflow

DB_PATH = BASE_DIR / "chinook.db"
settings = Settings.load()

print("数据库:", DB_PATH)
print("模型:", settings.model)
print("Base URL:", settings.base_url)
print("API Key:", "已配置" if settings.api_key else "未配置（复制 .env.example 为 .env 后填写）")


## 1. 数据库解析

读取表、字段、主键、外键、行数，以及每个字段的唯一值数、空值数、min/max、常见值。
这些信息会渲染成 Markdown，作为后续 prompt 的上下文。


In [ ]:
parser = DBParser(DB_PATH)
parser.parse_all()

print(parser.table_overview_markdown())
print()
print("外键关系：")
print(parser.relations_markdown())


In [ ]:
print(parser.table_schema_markdown("invoices"))


## 2-7. 完整工作流

配置了 API Key 后，`run()` 会依次执行：
table 选择 → SQL 生成 → 校验 →（失败则改写）→ 执行 → 结果汇总。
`on_step` 回调可以实时看到每一步的耗时和摘要。


In [ ]:
RUN_LLM = bool(settings.api_key)

if not RUN_LLM:
    print("未配置 API Key，跳过真实模型调用。")
    print("可以先运行 `python selftest_offline.py`，用假模型验证完整编排。")
else:
    def on_step(trace):
        detail = " ".join(trace.detail.splitlines())[:160]
        print(f"[{trace.name}] {trace.elapsed:.2f}s  {detail}")

    workflow = SQLAgentWorkflow(DB_PATH, settings=settings, on_step=on_step)
    result = workflow.run("哪个流派的曲目数量最多？列出前 5 个流派及其曲目数。")


In [ ]:
if RUN_LLM:
    import pandas as pd

    print(result.answer)
    print()
    print("SQL:")
    print(result.sql)
    pd.DataFrame(result.rows, columns=result.columns)


In [ ]:
if RUN_LLM:
    import pandas as pd

    pd.DataFrame([
        {"步骤": t.name, "耗时(s)": round(t.elapsed, 2), "说明": t.detail[:120]}
        for t in result.traces
    ])
    print("token 消耗：", result.llm_usage)


## 单独调用某一步

工作流的每一步都是公开方法，便于调试和替换：

```python
tables, reason = workflow.select_tables(question)
sql = workflow.generate_sql(question, tables)
ok, error = workflow.validate_sql(sql)
sql = workflow.repair_sql(question, tables, sql, error, history)
columns, rows, truncated = workflow.execute_sql(sql)
answer = workflow.summarize(question, sql, columns, rows, truncated)
```

不需要大模型也能使用数据库层：

```python
parser.validate_sql("DELETE FROM genres")      # (False, '只允许只读查询...')
parser.execute_sql("SELECT * FROM genres", max_rows=5)
```
